# 🔬 Cervexa - Cervical Lesion Instance Segmentation (YOLOv8-Seg)
### Pelatihan Model AI Segmentasi Kontur & Pelacakan Lesi Serviks (VIA)

Notebook Google Colab ini mengotomatiskan seluruh alur kerja pengembangan **Instance Segmentation** untuk Cervexa:
1. **Auto-Annotation**: Menganotasi otomatis 5.270 gambar Google Drive (Type 1, Type 2, Type 3) menggunakan algoritma deteksi lesi acetowhite dan FastSAM tanpa perlu menggambar manual.
2. **YOLOv8n-Seg Training**: Melatih model arsitektur lightweight YOLOv8 Nano Segmentation dengan GPU T4.
3. **Ekspor TFLite**: Mengonversi model menjadi `via_seg_model.tflite` (~7.5 MB) siap pasang di aplikasi Android Cervexa (HP & Smart TV).

---
### 📋 Panduan Singkat:
1. Pastikan GPU aktif di menu **Runtime > Change runtime type > T4 GPU**.
2. Jalankan **Step 1 s/d Step 8** secara berurutan.
3. Di akhir proses (Step 8), berkas `via_seg_model.tflite` akan otomatis terunduh ke komputer Anda.


## ⚙️ Step 1: Cek GPU & Instalasi Dependencies


In [ ]:
!nvidia-smi

!pip install -q ultralytics opencv-python-headless matplotlib
import os, sys, shutil, zipfile
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ PERINGATAN: GPU belum aktif! Aktifkan T4 GPU di Runtime > Change runtime type.')


## 📂 Step 2: Hubungkan Google Drive & Salin Dataset ke SSD Colab

Menyalin dataset dari Google Drive ke SSD lokal NVMe Colab (`/content/dataset_local`) untuk akselerasi pembacaan data hingga **50x lebih cepat**.


In [ ]:
from google.colab import drive
import os, sys, shutil, zipfile, time
from pathlib import Path

# Hubungkan Google Drive
print("⏳ Menghubungkan Google Drive...")
drive.mount('/content/drive', force_remount=True)

LOCAL_DIR = Path('/content/dataset_local')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
drive_dir = Path('/content/drive/MyDrive/Intel & MobileODT')

# Periksa apakah dataset sudah ada di SSD lokal Colab
existing_images = list(LOCAL_DIR.rglob('*.jpg')) + list(LOCAL_DIR.rglob('*.jpeg')) + list(LOCAL_DIR.rglob('*.png'))
if len(existing_images) > 500:
    print(f"⚡ Dataset lokal sudah siap! Ditemukan {len(existing_images)} file gambar di SSD lokal Colab.")
else:
    if not drive_dir.exists():
        raise FileNotFoundError("❌ Folder Google Drive '/content/drive/MyDrive/Intel & MobileODT' tidak ditemukan! Pastikan path folder di Google Drive benar.")

    print(f"🚀 Menyalin dataset dari '{drive_dir.name}' ke SSD Colab...")
    # Salin per item (menghindari crash FUSE socket / Transport endpoint is not connected)
    items = sorted(list(drive_dir.iterdir()))
    for idx, item in enumerate(items, 1):
        target_path = LOCAL_DIR / item.name
        print(f"[{idx}/{len(items)}] Menyalin {item.name} ...")
        if item.is_dir():
            shutil.copytree(item, target_path, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target_path)

    # Verifikasi jumlah file tersalin
    copied_images = list(LOCAL_DIR.rglob('*.jpg')) + list(LOCAL_DIR.rglob('*.jpeg')) + list(LOCAL_DIR.rglob('*.png'))
    copied_zips = list(LOCAL_DIR.rglob('*.zip'))
    print(f"\n📊 Status Salin: {len(copied_images)} gambar, {len(copied_zips)} zip file.")
    if len(copied_images) == 0 and len(copied_zips) == 0:
        raise RuntimeError("❌ GAGAL: Tidak ada file yang tersalin karena koneksi Google Drive terputus. Silakan lakukan 'Runtime > Restart session' dan jalankan ulang.")
    print("✅ Dataset berhasil disalin ke SSD Colab!")

# Ekstrak TYPE_2.zip jika ditemukan
for zip_f in LOCAL_DIR.rglob('*TYPE_2*.zip'):
    target_extract = LOCAL_DIR / 'TYPE_2'
    if not target_extract.exists() or len(list(target_extract.rglob('*.jpg'))) == 0:
        print(f"📦 Mengekstrak {zip_f.name}...")
        with zipfile.ZipFile(zip_f, 'r') as z:
            z.extractall(target_extract)
        print(f"✅ {zip_f.name} berhasil diekstrak!")
    break


## 🤖 Step 3: Auto-Annotation Batch (Bercak Acetowhite Optical Polygon)

Skrip ini mendeteksi area lesi serviks berbasis karakteristik optik VIA (reaksi warna putih asam asetat / *acetowhite lesion*) di sekitar ostium uteri, lalu mengekstrak poligon koordinat kurva `(x1, y1, x2, y2, ...)` yang dinormalisasi ke format label standar YOLO Segmentation:
`0 x1 y1 x2 y2 x3 y3 ... xn yn`

- Untuk gambar **NORMAL**: File label kosong (tidak ada lesi).
- Untuk gambar **ABNORMAL**: File label berisi poligon batas kontur lesi.


In [ ]:
import random

print('Memulai proses auto-annotation kontur lesi VIA (Color-Guided Optical Polygon Extraction)...')

YOLO_DATASET = Path('/content/yolo_dataset')
for split in ['train', 'val']:
    (YOLO_DATASET / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATASET / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Kumpulkan semua gambar
all_abnormal = []
all_normal = []

for root, dirs, files in os.walk('/content/dataset_local', followlinks=True):
    r_path = Path(root)
    name_lower = r_path.name.lower()
    imgs = [r_path / f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not imgs:
        continue
    if 'abnormal' in name_lower:
        all_abnormal.extend(imgs)
    elif 'normal' in name_lower:
        all_normal.extend(imgs)

print(f'Total gambar terdeteksi: {len(all_abnormal)} Abnormal, {len(all_normal)} Normal.')

def generate_lesion_polygon(img_bgr):
    h, w = img_bgr.shape[:2]
    # Deteksi area putih asam asetat (acetowhite) di ruang warna LAB
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    
    # Filter L (luminance) tinggi dan S moderat di sentral serviks
    mask_white = cv2.inRange(lab, np.array([145, 115, 115]), np.array([255, 145, 145]))
    
    # Masking bagian luar (fokus di radius 70% pusat serviks)
    center_y, center_x = h // 2, w // 2
    circ_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(circ_mask, (center_x, center_y), int(min(h, w) * 0.38), 255, -1)
    lesion_roi = cv2.bitwise_and(mask_white, circ_mask)
    
    contours, _ = cv2.findContours(lesion_roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
        
    largest_c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest_c) < (h * w * 0.008):
        return None
        
    # Sederhanakan poligon (approxPolyDP) menjadi 15-30 titik agar ringan
    epsilon = 0.006 * cv2.arcLength(largest_c, True)
    approx = cv2.approxPolyDP(largest_c, epsilon, True)
    
    # Normalisasi koordinat ke 0.0 - 1.0
    pts = []
    for p in approx:
        x_norm = float(p[0][0]) / w
        y_norm = float(p[0][1]) / h
        pts.extend([f'{x_norm:.5f}', f'{y_norm:.5f}'])
        
    if len(pts) < 6:
        return None
    return '0 ' + ' '.join(pts)

print('Memproses auto-annotation & penyusunan dataset YOLO...')
dataset_items = []
for p in all_abnormal:
    dataset_items.append((p, True))
for p in all_normal:
    dataset_items.append((p, False))

random.seed(42)
random.shuffle(dataset_items)

val_count = int(len(dataset_items) * 0.15)
train_items = dataset_items[val_count:]
val_items = dataset_items[:val_count]

def export_split(items, split_name):
    saved = 0
    for img_path, is_abnormal in items:
        try:
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            dest_img = YOLO_DATASET / 'images' / split_name / f'{split_name}_{saved:05d}.jpg'
            dest_txt = YOLO_DATASET / 'labels' / split_name / f'{split_name}_{saved:05d}.txt'
            
            cv2.imwrite(str(dest_img), img)
            
            if is_abnormal:
                poly = generate_lesion_polygon(img)
                if poly:
                    dest_txt.write_text(poly)
                else:
                    dest_txt.write_text('')
            else:
                dest_txt.write_text('')
            saved += 1
        except Exception:
            pass
    print(f'✅ Berhasil menyiapkan {saved} gambar untuk {split_name}')

export_split(train_items, 'train')
export_split(val_items, 'val')


## 📄 Step 4: Buat Konfigurasi Dataset (`data.yaml`)


In [ ]:
yaml_text = """path: /content/yolo_dataset
train: images/train
val: images/val

names:
  0: abnormal_lesion
"""

with open('/content/yolo_dataset/data.yaml', 'w') as f:
    f.write(yaml_text.strip())

print('File data.yaml berhasil dibuat!')


## 🚀 Step 5: Pelatihan Model YOLOv8n-Seg (T4 GPU)

Melatih arsitektur **YOLOv8 Nano Segmentation** (`yolov8n-seg.pt`) dengan resolusi `imgsz=384` yang optimal untuk performa real-time di Android.


In [ ]:
from ultralytics import YOLO

# Load pre-trained lightweight segmentation weights
model = YOLO('yolov8n-seg.pt')

# Mulai pelatihan
results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=40,
    imgsz=384,
    batch=16,
    device=0,
    workers=4,
    name='cervexa_yolo_seg',
    save=True,
    plots=True
)
print('✅ Training YOLOv8n-Seg selesai!')


## 📊 Step 6: Validasi & Visualisasi Hasil Deteksi Kontur


In [ ]:
# Jalankan validasi pada dataset val
metrics = model.val()
print('Hasil Metrik Segmentasi:')
print(f'Mask mAP50-95: {metrics.seg.map:.4f}')
print(f'Mask mAP50:    {metrics.seg.map50:.4f}')

# Tampilkan salah satu contoh hasil prediksi
import glob
val_imgs = glob.glob('/content/yolo_dataset/images/val/*.jpg')
if val_imgs:
    sample_preds = model.predict(val_imgs[:3], imgsz=384)
    for i, pred in enumerate(sample_preds):
        res_plot = pred.plot()
        plt.figure(figsize=(6, 6))
        plt.imshow(cv2.cvtColor(res_plot, cv2.COLOR_BGR2RGB))
        plt.title(f'Prediksi Kontur Lesi Serviks #{i+1}')
        plt.axis('off')
        plt.show()


## 📦 Step 7: Ekspor ke Format TFLite (`via_seg_model.tflite`)

Mengonversi bobot PyTorch ke format TensorFlow Lite (`.tflite`) berukuran **~7.5 MB** yang kompatibel langsung dengan Android Cervexa.


In [ ]:
best_pt = Path('/content/runs/segment/cervexa_yolo_seg/weights/best.pt')
if not best_pt.exists():
    best_pt = Path('/content/runs/segment/cervexa_yolo_seg2/weights/best.pt')

print('Mengonversi model YOLO-Seg ke format TensorFlow Lite...')
trained_yolo = YOLO(str(best_pt))
tflite_path = trained_yolo.export(format='tflite', imgsz=384)
print(f'✅ Berhasil diekspor: {tflite_path}')

# Salin ke nama resmi
dest_official = Path('/content/via_seg_model.tflite')
exported_files = list(Path('/content/runs/segment').rglob('*.tflite'))
if exported_files:
    shutil.copy(exported_files[0], dest_official)
    sz_mb = dest_official.stat().st_size / (1024 * 1024)
    print(f'📁 Berkas model siap unduh: {dest_official} ({sz_mb:.2f} MB)')


## ⬇️ Step 8: Unduh Otomatis Model `via_seg_model.tflite`


In [ ]:
from google.colab import files

target_file = '/content/via_seg_model.tflite'
if os.path.exists(target_file):
    print('Mengunduh via_seg_model.tflite ke komputer Anda...')
    files.download(target_file)
else:
    print('File tflite belum ditemukan di /content/via_seg_model.tflite. Periksa hasil Step 7.')
